In [1]:
import sys
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import datasets
from functools import partial

import torch

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
model_path = "../../self-corrective-llama_untrained"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model_config = AutoConfig.from_pretrained(model_path)
model_config.deletion_threshold = 0.5

model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True, config=model_config)

In [3]:
dataset = datasets.load_from_disk("../../dataset/training")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 31519
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 3503
})


In [4]:
sample = train_dataset[0]
sample["input_ids"] = torch.tensor([sample["input_ids"][390:420]])
sample["attention_mask"] = torch.tensor([sample["attention_mask"][390:420]])
sample["labels"] = torch.tensor([sample["labels"][390:420]])
sample["hallucination_labels"] = torch.tensor([sample["hallucination_labels"][390:420]])
print(sample)

{'input_ids': tensor([[ 7616,    82,   315, 39881,   489,   393,  7616,    82,   315,  5684,
         25485,   340,    28,   220,   508,   482,   320,   605,   489,   220,
            20,   489,   220,    19,   340,    28,   220,   508,   482,   220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ 7616,    82,   315, 39881,   489,   393,  7616,    82,   315,  5684,
         25485,   340,    28,   220,   508,   482,   320,   605,   489,   220,
            20,   489,   220,    19,   340,    28,   220,   508,   482,   220]]), 'hallucination_labels': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]])}


In [5]:
# model.forward(
#     input_ids=sample["input_ids"], 
#     attention_mask=sample["attention_mask"], 
#     labels=sample["labels"], 
#     hallucination_labels=sample["hallucination_labels"]
# )

In [6]:
text = "What is the capital of France?"
sample = train_dataset[0]
input_ids = tokenizer.encode(text)
input_ids = torch.tensor([input_ids])
attention_mask = torch.ones_like(input_ids)

# input_ids = torch.tensor([sample["input_ids"]])
# attention_mask = torch.ones_like(torch.tensor([sample["attention_mask"]]))

result = model.generate(input_ids=input_ids, tokenizer=tokenizer)

Del S IDs: tensor([[   58,  4644,  3766, 11914,    60]])
torch.Size([1, 5])
Del A IDs: tensor([[  58, 4644,  682, 1495,   60]])
torch.Size([1, 5])
Processing initial prompt...


Next token logits: torch.Size([1, 128256])
tensor([[ 5.0945,  2.6669,  4.5030,  ..., -0.0846, -0.0852, -0.0852]])
Hallucination logits: torch.Size([1, 3])
tensor([[-0.3589, -0.5549, -0.0346]])
Past key values length: 16
Hallucination probs: torch.Size([1, 3])
tensor([[0.3120, 0.2565, 0.4315]])
Sampling from the main logits.
Current tokens: torch.Size([1, 1])
tensor([[12366]])
Generated IDs: torch.Size([1, 9])
tensor([[128000,   3923,    374,    279,   6864,    315,   9822,     30,  12366]])


Next token logits: torch.Size([1, 128256])
tensor([[17.1726,  8.3706,  7.8917,  ..., -0.1116, -0.1114, -0.1114]])
Hallucination logits: torch.Size([1, 3])
tensor([[ 0.2122, -0.4430, -0.9621]])
Past key values length: 16
Hallucination probs: torch.Size([1, 3])
tensor([[0.5470, 0.2840, 0.1690]])
Sampling from the main logits

In [7]:
res = tokenizer.decode(result[0])
print(res)

<|begin_of_text|>What is the capital of France? Paris
The[delete all text] capital of France is[delete all text] Paris.<|eot_id|>
